# ID6004W Industrial Vision - Assignment 1
### Lectures 01-02: Image Filtering and Convolution

| | |
|---|---|
| **Released** | Oct 1st 2026 |
| **Due** | Oct 21st 2026 (5.00 PM IST) |
| **Total** | 100 marks (will be scaled down appropriately later) |

## Instructions

**Working in Colab.** Open this notebook in Google Colab and immediately use *File → Save a copy in Drive*; work only in your copy. Type your answers in the cells marked **Your answer**, directly below each question (double-click a cell to edit it). Mathematics can be typed in LaTeX between `$...$` or `$$...$$`. Photographs of hand-written working are **not** accepted in place of typed answers.

- **The writing must be your own.**
- Cells marked **Optional check** are not graded. Use them to verify a hand calculation numerically, but the answer you submit must be derived by hand with the steps shown.
-  **Submission.** When you are done, use *File → Download → Download .ipynb*, rename the file to `<ROLLNO>_assignment1.ipynb`, and upload it here: [SUBMISSION_LINK](https://forms.gle/pRwExrMB1dvKoU138). For queries contact Rishabh Shukla (da26e011@smail.iitm.ac.in). 
- **Late policy:** 10 marks will be deducted for each day the assignment is submitted late after the due date.

## Student details

- **Name:**
- **Roll number:**

In [2]:
# Imports used by the optional check cells.
import numpy as np
from scipy import ndimage
rng = np.random.default_rng(0)

---
## Question 1. What is, and what is not, a convolution  [25 marks]

In Lecture 01 we proved that linearity together with shift invariance forces an operator to be a convolution. This question asks you to use that theorem as a *test*.

**(a)** For each operator $\mathcal{T}$ below, state whether it is (i) linear and (ii) shift invariant. If it is both, give its impulse response $h$ explicitly (as a small array, or as a formula). Here $f$ is the input image and $g=\mathcal{T}\{f\}$. **[15]**

1. $g(m,n) = f(m,n) - f(m-1,n)$
2. $g(m,n) = \operatorname{median}\big\{f(m+i,n+j) : |i|\le 1, |j|\le 1\big\}$
3. $g(m,n) = c(m,n)\,f(m,n)$, where $c$ is a fixed, non-constant mask (e.g. lens vignetting)
4. $g(m,n) = f(m,n)^{2}$
5. $g(m,n) = f(n,m)$ (transpose the image)
6. $g(m,n) = f(m,n) + 10$ (brighten)

**Your answer 1(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [3]:
# Optional check for 1(a). The six operators, with periodic boundaries so that shifts are exact.
N = 16
c_mask = rng.uniform(0.5, 1.0, (N, N))          # a fixed, non-constant mask

T = {
    1: lambda f: f - np.roll(f, 1, axis=0),
    2: lambda f: ndimage.median_filter(f, size=3, mode="wrap"),
    3: lambda f: c_mask * f,
    4: lambda f: f ** 2,
    5: lambda f: f.T,
    6: lambda f: f + 10,
}

def shift(f, dm, dn):
    return np.roll(f, (dm, dn), axis=(0, 1))

# TODO (optional): write is_linear(op) and is_shift_invariant(op) that test the two properties on
# random images f1, f2 and random scalars a, b, and print the result for every operator in T.

**(b)** Let $\mathcal{T}_1$ and $\mathcal{T}_2$ be two LSI systems with impulse responses $h_1$ and $h_2$. Show that the cascade $\mathcal{T}_2\{\mathcal{T}_1\{f\}\}$ is again LSI, identify its impulse response, and state why the order of the two systems does not matter. **[10]**

**Your answer 1(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 2. Separability and rank  [25 marks]

**(a)** Determine the rank of each kernel below, and for those of rank one give an explicit factorisation $H=h_1h_2^{\top}$. Do this by hand, using the fact that a nonzero matrix has rank one exactly when every $2\times2$ minor vanishes. **[15]**

$$
A=\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
B=\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}
\quad
C=\begin{bmatrix}1&2&3\\2&4&6\\3&6&9\end{bmatrix}
\quad
D=\begin{bmatrix}1&1&1\\1&2&1\\1&1&1\end{bmatrix}
$$

**Your answer 2(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [4]:
# Optional check for 2(a).
A = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16
B = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
C = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]])
D = np.array([[1, 1, 1], [1, 2, 1], [1, 1, 1]])

# TODO (optional): print the rank of each kernel, and for the rank-one kernels check that your
# factorisation np.outer(h1, h2) reproduces the kernel.

**(b)** An $N\times N$ image is smoothed with a Gaussian of $\sigma=5$, truncated to $2\lceil 3\sigma\rceil+1$ taps. State the kernel size $M$, the number of multiplications per pixel for a direct 2D pass and for two 1D passes, and the resulting speed-up. **[10]**

**Your answer 2(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 3. Noise gain  [25 marks]

Let the noise be i.i.d. with variance $\sigma_\eta^{2}$, so that a linear filter $w$ produces output noise variance $\sigma_\eta^{2}\sum_k w_k^{2}$.

**(a)** Compute $\sum_k w_k^{2}$ for each of the four kernels below, and give the output noise standard deviation as a multiple of $\sigma_\eta$. **[10]**

$$
\text{box } \tfrac19\begin{bmatrix}1&1&1\\1&1&1\\1&1&1\end{bmatrix}
\quad
\text{Gaussian } \tfrac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}
\quad
\text{Sobel } \begin{bmatrix}1&0&-1\\2&0&-2\\1&0&-1\end{bmatrix}
\quad
\text{sharpen } \begin{bmatrix}0&-1&0\\-1&5&-1\\0&-1&0\end{bmatrix}
$$

**Your answer 3(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [5]:
# Optional check for 3(a): filter pure noise with each kernel and measure the output standard deviation.
kernels = {
    "box":      np.ones((3, 3)) / 9,
    "Gaussian": np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16,
    "Sobel":    np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]]),
    "sharpen":  np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}
sigma_eta = 1.0
noise = rng.normal(0, sigma_eta, (1024, 1024))

# TODO (optional): for each kernel compute sum(w**2), predict the output std, and compare it with the
# std of ndimage.convolve(noise, w, mode="wrap").

**(b)** Prove that among *all* $3\times3$ kernels whose weights sum to $1$, the box kernel minimises $\sum_k w_k^{2}$. (One line of Cauchy-Schwarz.) **[5]**

**Your answer 3(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(c)** Your answers to (a) and (b) together say that the box filter is a strictly better denoiser, per pixel of support, than the $3\times3$ Gaussian. Yet the Gaussian is mostly the preferred choice. Give one distinct reason why. **[5]**

**Your answer 3(c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(d)** An alternative to spatial smoothing is to average $L$ frames of a static scene. How many frames are needed to match the noise reduction of one $3\times3$ box filter, and what does frame averaging give you that the box filter cannot? **[5]**

**Your answer 3(d).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 4. Zero-crossings  [25 marks]

**(a)** Show that the stencil $[1\;\;{-}2\;\;1]$ annihilates every affine ramp $f(x)=ax+b$, and that the five-point $\nabla^{2}$ annihilates every plane $f(x,y)=ax+by+c$. What does this tell you about the response of $\nabla^{2}$ inside a smoothly shaded region, and why is that desirable? **[15]**

**Your answer 4(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [6]:
# Optional check for 4(a).
stencil_1d = np.array([1, -2, 1])
laplacian_5pt = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
x = np.arange(20.0)
X, Y = np.meshgrid(np.arange(20.0), np.arange(20.0))

# TODO (optional): apply the stencil to a ramp a*x + b and the five-point Laplacian to a plane
# a*X + b*Y + c (use mode="valid" / interior pixels only) and check that the result is zero.

**(b)** A 1D edge is sampled and the second difference takes the values

$$f''(62)=+1.07,\qquad f''(63)=+0.33,\qquad f''(64)=-0.49 .$$

Locate the zero-crossing to sub-pixel accuracy by linear interpolation, showing the arithmetic. The corresponding $|f'|$ values are $14.10,\,14.80,\,14.72$; state the best estimate of the edge location obtainable from $\arg\max|f'|$ alone. **[10]**

**Your answer 4(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [7]:
# Optional check for 4(b).
xs = np.array([62, 63, 64])
f2 = np.array([1.07, 0.33, -0.49])     # second difference
f1 = np.array([14.10, 14.80, 14.72])   # |first difference|

# TODO (optional): compute the linearly interpolated zero-crossing of f2 and xs[argmax(f1)].

---
*End of Assignment 1.*